In [1]:
!pip install -q huggingface_hub datasets

In [2]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)
print(whoami()["name"])

thkt2110


In [3]:
from pathlib import Path
import json
import os

repo_id = "nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim"
local_dir = Path("/kaggle/working/gr00t_x_embodiment_sim")
report_path = Path("/kaggle/working/download_report_6_10.json")

# Subsets 6-10 in the GR-1 priority list from the CK plan.
SUBSETS_6_10 = [
    "gr1_arms_waist.CanToDrawer", 
]

def folder_size_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    total = sum(p.stat().st_size for p in path.rglob("*") if p.is_file())
    return total / (1024 ** 3)

print("Target subsets:")
for i, subset in enumerate(SUBSETS_6_10, start=6):
    print(f"  {i}. {subset}")

print(f"\nLocal output: {local_dir}")
print(f"Report output: {report_path}")

Target subsets:
  6. gr1_arms_waist.CanToDrawer

Local output: /kaggle/working/gr00t_x_embodiment_sim
Report output: /kaggle/working/download_report_6_10.json


In [4]:
from huggingface_hub import snapshot_download
import time

local_dir.mkdir(parents=True, exist_ok=True)


for subset_name in SUBSETS_6_10:
    subset_path = local_dir / subset_name
    print("\n" + "=" * 80)
    print(f"Downloading subset: {subset_name}")

    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        allow_patterns = [f"{subset_name}/meta/**",f"{subset_name}/data/**",],
        local_dir=str(local_dir),
        max_workers=3,
    )

    print(f"Downloaded to: {subset_path}")
    print(f"Subset size: {folder_size_gb(subset_path):.2f} GB")
    time.sleep(15)

print("\nDownload step finished.")

Fetching ... files: 0it [00:00, ?it/s]

Downloaded to: /kaggle/working/gr00t_x_embodiment_sim/gr1_arms_waist.CanToDrawer
Subset size: 1.49 GB

Download step finished.


In [5]:
report = {
    "repo_id": repo_id,
    "local_dir": str(local_dir),
    "subsets": [],
}

for subset_name in SUBSETS_6_10:
    subset_path = local_dir / subset_name
    data_dir = subset_path / "data"
    videos_dir = subset_path / "videos"
    meta_dir = subset_path / "meta"

    item = {
        "name": subset_name,
        "path": str(subset_path),
        "exists": subset_path.exists(),
        "size_gb": round(folder_size_gb(subset_path), 3),
        "has_data_dir": data_dir.exists(),
        "has_videos_dir": videos_dir.exists(),
        "has_meta_dir": meta_dir.exists(),
        "num_parquet_files": len(list(data_dir.rglob("*.parquet"))) if data_dir.exists() else 0,
        "num_video_files": len(list(videos_dir.rglob("*.mp4"))) if videos_dir.exists() else 0,
    }
    report["subsets"].append(item)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nDisk usage:")
os.system("df -h /kaggle/working")
os.system("du -sh /kaggle/working/gr00t_x_embodiment_sim 2>/dev/null || true")
print(f"\nSaved report to: {report_path}")

{
  "repo_id": "nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim",
  "local_dir": "/kaggle/working/gr00t_x_embodiment_sim",
  "subsets": [
    {
      "name": "gr1_arms_waist.CanToDrawer",
      "path": "/kaggle/working/gr00t_x_embodiment_sim/gr1_arms_waist.CanToDrawer",
      "exists": true,
      "size_gb": 1.494,
      "has_data_dir": true,
      "has_videos_dir": false,
      "has_meta_dir": true,
      "num_parquet_files": 10107,
      "num_video_files": 0
    }
  ]
}

Disk usage:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  1.6G   18G   8% /kaggle/working
1.6G	/kaggle/working/gr00t_x_embodiment_sim

Saved report to: /kaggle/working/download_report_6_10.json
